# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [15]:
# Install required packages
import subprocess
import sys

packages_to_install = [
    "chromadb>=1.0.4",
    "openai>=1.73.0",
    "pydantic>=2.11.3",
    "python-dotenv>=1.1.0",
    "tavily-python>=0.5.4"
]

for package in packages_to_install:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"✗ Failed to install {package}: {e}")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


✓ chromadb>=1.0.4 installed


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


✓ openai>=1.73.0 installed


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


✓ pydantic>=2.11.3 installed


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


✓ python-dotenv>=1.1.0 installed
✓ tavily-python>=0.5.4 installed


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# DONE: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [ ]:
load_dotenv()

True

### VectorDB Instance

In [ ]:
# Instantiate ChromaDB Persistent Client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

### Collection

In [ ]:
# Initialize OpenAI embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

In [ ]:
# Create or get collection with embedding function
collection = chroma_client.get_or_create_collection(
    name="games",
    embedding_function=embedding_fn
)

print(f"Collection '{collection.name}' created/retrieved successfully")

Collection 'games' created/retrieved successfully


### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Verify Collection and Test Semantic Search

In [11]:
# Verify collection count and test semantic search
print(f"Total documents in collection: {collection.count()}")

# Test semantic search with sample queries
test_queries = [
    "FIFA soccer game",
    "God of War",
    "Pokémon Red"
]

print("\n" + "="*60)
print("SEMANTIC SEARCH TEST RESULTS")
print("="*60)

for query in test_queries:
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    
    if results and results['documents']:
        for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
            confidence = 1 - distance  # Normalize distance to confidence
            print(f"  Result {i+1} (confidence: {confidence:.2f})")
            print(f"    Document: {doc[:80]}...")
            if results['metadatas'] and results['metadatas'][0]:
                metadata = results['metadatas'][0][i]
                print(f"    Game: {metadata.get('Name', 'N/A')}")
                print(f"    Platform: {metadata.get('Platform', 'N/A')}")
        print(f"  Best match distance: {results['distances'][0][0]:.4f}")
    else:
        print("  No results found")

print("\n" + "="*60)
print("✓ RAG Pipeline Setup Complete!")
print("="*60)

Total documents in collection: 15

SEMANTIC SEARCH TEST RESULTS

Query: 'FIFA soccer game'
------------------------------------------------------------
  Result 1 (confidence: 0.36)
    Document: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuri...
    Game: Gran Turismo 5
    Platform: PlayStation 3
  Result 2 (confidence: 0.33)
    Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a w...
    Game: Gran Turismo
    Platform: PlayStation 1
  Result 3 (confidence: 0.33)
    Document: [Wii] Wii Sports (2006) - A collection of sports games that utilize the Wii's mo...
    Game: Wii Sports
    Platform: Wii
  Best match distance: 0.6367

Query: 'God of War'
------------------------------------------------------------
  Result 1 (confidence: 0.33)
    Document: [PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that l...
    Game: Marvel's Spider-Man
    Platform: PlayStation 4
  Result 2 (confidence:

## Part 2 - Agent Implementation

Build an AI agent with 3 tools: retrieve_game, evaluate_retrieval, and game_web_search

In [12]:
# Import required libraries for agent implementation
import json
from pydantic import BaseModel
from tavily import TavilyClient
from openai import OpenAI

# Initialize OpenAI client using LLM wrapper if possible, otherwise direct
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_BASE_URL"))

print("✓ Agent libraries imported successfully")

✓ Agent libraries imported successfully


/Users/daniel.a.robles/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Tool 1: Retrieve Game

In [13]:
def retrieve_game(query):
    """
    Retrieves games from ChromaDB based on semantic similarity to the query.
    
    Args:
        query (str): A question about games or game-related topics
        
    Returns:
        dict with retrieved games, confidence scores, and metadata
    """
    # Query ChromaDB collection
    results = collection.query(query_texts=[query], n_results=3)
    
    retrieved_games = []
    if results['documents'] and len(results['documents']) > 0:
        for i, doc in enumerate(results['documents'][0]):
            distance = results['distances'][0][i] if results['distances'] else 0
            confidence = 1 - distance  # Convert distance to confidence
            metadata = results['metadatas'][0][i] if results['metadatas'] else {}
            
            retrieved_games.append({
                'document': doc,
                'platform': metadata.get('platform', 'Unknown'),
                'name': metadata.get('name', 'Unknown'),
                'year': metadata.get('year', 'Unknown'),
                'confidence': round(confidence, 2),
                'distance': round(distance, 2)
            })
    
    return {
        'results': retrieved_games,
        'query': query,
        'num_results': len(retrieved_games)
    }

# Test retrieve_game
test_retrieval = retrieve_game("Pokémon games")
print("✓ retrieve_game tool working")
print(f"  Found {test_retrieval['num_results']} results")

✓ retrieve_game tool working
  Found 3 results


### Tool 2: Evaluate Retrieval

In [14]:
def evaluate_retrieval(query, retrieval_result):
    """
    Uses an LLM to evaluate if retrieved documents are sufficient to answer the query.
    
    Args:
        query (str): The original user question
        retrieval_result (dict): Results from retrieve_game tool
        
    Returns:
        dict with evaluation report including relevance and confidence
    """
    # Format documents for evaluation
    documents_text = "\n".join([
        f"- {r['name']} ({r['platform']}, {r['year']}): {r['document'][:100]}..."
        for r in retrieval_result.get('results', [])
    ])
    
    # Create evaluation prompt
    eval_prompt = f"""Your task is to evaluate if the retrieved documents are sufficient to answer the user's question.

User Question: {query}

Retrieved Documents:
{documents_text}

Analyze whether these documents contain enough information. Consider:
1. Do the documents directly address the question?
2. Is the information specific enough?
3. Your confidence level (0.0 to 1.0)?

Respond ONLY with a JSON object containing:
{{"is_relevant": true/false, "confidence": 0.0-1.0, "reasoning": "brief explanation"}}"""
    
    try:
        # Use LLM to evaluate
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": eval_prompt}],
            temperature=0.3
        )
        
        eval_text = response.choices[0].message.content
        
        # Extract and parse JSON
        json_start = eval_text.find('{')
        json_end = eval_text.rfind('}') + 1
        if json_start != -1 and json_end > json_start:
            json_str = eval_text[json_start:json_end]
            eval_data = json.loads(json_str)
            return {
                'is_relevant': eval_data.get('is_relevant', True),
                'confidence': float(eval_data.get('confidence', 0.5)),
                'reasoning': eval_data.get('reasoning', '')
            }
    except Exception as e:
        pass
    
    # Default response
    return {
        'is_relevant': True,
        'confidence': 0.6,
        'reasoning': 'Evaluation completed with fallback response'
    }

# Test evaluate_retrieval
test_eval = evaluate_retrieval("When was Pokémon Gold released?", test_retrieval)
print("✓ evaluate_retrieval tool working")
print(f"  Relevance: {test_eval['is_relevant']}")
print(f"  Confidence: {test_eval['confidence']}")

✓ evaluate_retrieval tool working
  Relevance: True
  Confidence: 0.8


### Tool 3: Game Web Search

In [15]:
def game_web_search(query):
    """
    Searches the web for game-related information using Tavily API.
    
    Args:
        query (str): Search query about games
        
    Returns:
        dict with search answer, sources, and confidence
    """
    try:
        tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        search_response = tavily_client.search(query, max_results=5)
        
        answer = search_response.get("answer", "No answer found")
        sources = [result.get("url", "") for result in search_response.get("results", [])]
        
        return {
            'answer': answer or "No answer found",
            'sources': sources[:3],  # Limit to top 3 sources
            'confidence': 0.8,
            'query': query,
            'success': True
        }
    except Exception as e:
        return {
            'answer': f"Web search error: {str(e)}",
            'sources': [],
            'confidence': 0.0,
            'query': query,
            'success': False,
            'error': str(e)
        }

# Test game_web_search
test_search = game_web_search("Pokémon Gold and Silver release date")
print("✓ game_web_search tool working")
answer_preview = test_search['answer'][:100] if test_search['answer'] else "No answer"
print(f"  Answer: {answer_preview}...")
print(f"  Success: {test_search['success']}")
print(f"  Sources found: {len(test_search['sources'])}")

✓ game_web_search tool working
  Answer: No answer found...
  Success: True
  Sources found: 3


## Agent Implementation

The agent orchestrates the three tools to answer gaming questions using RAG with fallback to web search

In [16]:
class UdaPlayAgent:
    """
    Gaming AI Analytics Agent with retrieval, evaluation, and web search capabilities
    """
    
    def __init__(self, model="gpt-4o-mini", temperature=0.3):
        self.model = model
        self.temperature = temperature
        self.system_prompt = """You are an expert gaming AI assistant specializing in video game information and history.

Your role is to answer questions about video games using three tools:
1. retrieve_game - Search the internal game database
2. evaluate_retrieval - Judge if retrieved results answer the question
3. game_web_search - Search the web for current/missing information

Workflow:
1. First, always try retrieve_game to query the internal database
2. Then use evaluate_retrieval to check if results are sufficient
3. If evaluation shows low confidence or irrelevant results, use game_web_search
4. Provide clear, cited answers with platform, year, and publisher info
5. Always explain which tool you used and why

Format your final answer clearly with:
- Answer to the question
- Sources (database or web)
- Confidence level
- Key details (platform, year, publisher if relevant)"""
    
    def _call_tool(self, tool_name, query, retrieval_result=None):
        """Call a specific tool"""
        if tool_name == "retrieve_game":
            return retrieve_game(query)
        elif tool_name == "evaluate_retrieval":
            return evaluate_retrieval(query, retrieval_result)
        elif tool_name == "game_web_search":
            return game_web_search(query)
        return None
    
    def invoke(self, query, session_id=None):
        """
        Main agent invocation - orchestrates tools to answer the query
        """
        print(f"\n{'='*70}")
        print(f"Query: {query}")
        print(f"{'='*70}")
        
        # Step 1: Retrieve from internal database
        print("\n1. Retrieving from game database...")
        retrieval = self._call_tool("retrieve_game", query)
        print(f"   Found {retrieval['num_results']} results")
        for i, result in enumerate(retrieval['results'], 1):
            print(f"   {i}. {result['name']} ({result['platform']}, {result['year']}) - confidence: {result['confidence']}")
        
        # Step 2: Evaluate retrieval quality
        print("\n2. Evaluating retrieval quality...")
        evaluation = self._call_tool("evaluate_retrieval", query, retrieval)
        print(f"   Relevant: {evaluation['is_relevant']}")
        print(f"   Confidence: {evaluation['confidence']}")
        print(f"   Reasoning: {evaluation['reasoning']}")
        
        # Step 3: Decide if web search needed
        use_web_search = not evaluation['is_relevant'] or evaluation['confidence'] < 0.5
        
        if use_web_search:
            print("\n3. Confidence too low, using web search...")
            web_search = self._call_tool("game_web_search", query)
            print(f"   Web search completed")
            
            # Compile final answer
            final_answer = f"""
ANSWER (from Web Search):
{web_search['answer']}

SOURCES:
{', '.join(web_search['sources'][:2]) if web_search['sources'] else 'No sources found'}

CONFIDENCE: {web_search['confidence']}

REASONING: Database confidence was too low ({evaluation['confidence']}), so I searched the web for current information."""
        else:
            print("\n3. Database results are sufficient, using retrieval...")
            # Compile final answer from database
            top_result = retrieval['results'][0] if retrieval['results'] else None
            if top_result:
                final_answer = f"""
ANSWER (from Game Database):
{top_result['document']}

GAME DETAILS:
- Name: {top_result['name']}
- Platform: {top_result['platform']}
- Year: {top_result['year']}
- Confidence: {top_result['confidence']}

REASONING: Found directly in the game database with high confidence."""
            else:
                final_answer = "No information found in database or web."
        
        print(f"\n{'='*70}")
        print("FINAL ANSWER:")
        print(f"{'='*70}")
        print(final_answer)
        
        return final_answer

# Create agent instance
agent = UdaPlayAgent(model="gpt-4o-mini", temperature=0.3)
print("✓ UdaPlayAgent created successfully")

✓ UdaPlayAgent created successfully


## Test Agent with Example Queries

In [18]:
# Define test queries as per requirements
test_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

print("RUNNING AGENT ON TEST QUERIES")
print("=" * 70)

for i, query in enumerate(test_queries, 1):
    print(f"\n\n[TEST QUERY {i}/{len(test_queries)}]")
    try:
        response = agent.invoke(query, session_id=f"test_session_{i}")
    except Exception as e:
        print(f"Error executing query: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'='*70}")
print("✓ ALL TESTS COMPLETED")
print("="*70)

RUNNING AGENT ON TEST QUERIES


[TEST QUERY 1/3]

Query: When was Pokémon Gold and Silver released?

1. Retrieving from game database...
   Found 3 results
   1. Unknown (Unknown, Unknown) - confidence: 0.66
   2. Unknown (Unknown, Unknown) - confidence: 0.5
   3. Unknown (Unknown, Unknown) - confidence: 0.24

2. Evaluating retrieval quality...
   Relevant: True
   Confidence: 0.9
   Reasoning: One of the retrieved documents explicitly states that Pokémon Gold and Silver were released in 1999, directly answering the user's question. The information is specific and relevant.

3. Database results are sufficient, using retrieval...

FINAL ANSWER:

ANSWER (from Game Database):
[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.

GAME DETAILS:
- Name: Unknown
- Platform: Unknown
- Year: Unknown
- Confidence: 0.66

REASONING: Found directly in the game database with high confidence.


[TEST QUERY 2/3]

Qu